intro_to_Language_modeling final
-----

<ins>Goal</ins>
Build a character language model to predict next character

create a neuralnet with


1. Init the env and download training dataset  
  * Add necessary imports - torch,matplotlib, random
  * download the names dataset
  * read the download file and extract names to a list
2. Build vocabulary of characters
  * get all chars in the dataset and sort them alphabetically
  * create a lookup dict for char to index
  * create a lookup dict for index to char
  * set the vocab_size constant
3. Build training,dev and test split from dataset
  * iterate over the words and build thier numerical input and output equivalent - where  input is a numerical representation of sequence of block_size characters and output is numerical representation of next character
  * shuffle and split above input and output into training,dev and test split
4. Create custom classe for neural net components
  * Sequential - to store list of layers and execute them sequentially
  * Linear class - to perform linear transformation on the input data.
    * accept fan_in, fan_out & bias flag parameters
    * set rand initial weights (optionally use Kaiming init)
    * set zero as initial biases
    * calculate out - conditionally add bias to the calculation if bias flag is set(ignore it for cases where we use BatchNorm, because we use bias from BatchNorm)
    * add method to returns weight and bias parameters
  * BatchNorm1d - Normalizes the input features across the batch dimension. This is done by calculates the mean and variance of the input for the current batch and uses these to normalize the data
    * accept dim, eps & momentum parameters
    * set ones as initial gamma values
    * set zeros as initial beta values
    * conditional calculate the batch-mean and batch-variance
    * calculate the output - gamma * normalized-data + beta
    * conditional calculate running batch-mean and batch-variance during training
    * add method to returns learnable parameters(gamma and beta)
  * Tanh - Activation functions introduce non-linearity into the network, allowing it to learn more complex, non-linear relationships in the data
  * Embedding - convert input(scalar index) into a vector in a higher-dimensional space
  * FlattenConsecutive - faltten the input tensor to different shape, effectively grouping(ngram like) consecutive elements in the sequence dimension
5. Initialize a network
  * define the no of neurons in hidden layer - 200
  * define the input embedding dimension - 10
  * define the mode with its layers
  * make last layer(output) a little less confident
  * mark grad required to all parameters
6. Traing the model
  * Randomly selects indices for a batch of training data
  * do the forward pass and calculate the loss
  * invoke backpass
  * update the parameters (use step learning rate decay) - use SDG-stochastic gradient descent
  * capture loss in each training cycle for stats
7. plot the  captured loss graph
8. evaluate the loss against training and test split
  * ensure that grad is turned off (switch off training flag to false)
9. Sample from model
  * start with context [0,0,0] that reprsents chars'...'
  * provide this context to model and get the logits
  * find the probablity distribution for the logits
  * use the probablity distribution get a sample index for char
  * increcemt the context by appendeing the above sampled index
  * continue until get 0(index for char '.') as the sampled index



Initial Setup
----

In [77]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as ply
import random
%matplotlib inline

In [78]:
# download the sample data
!curl https://raw.githubusercontent.com/karpathy/makemore/refs/heads/master/names.txt -o names.txt -s


In [79]:
# extract the names from the downloaded file
words = open('names.txt', 'r').read().splitlines()
print("Total words in sample dataset",len(words))

Total words in sample dataset 32033


In [80]:
## build vocab for characters ##

end_char = '.'

#get all chars in the dataset and sort them alphabetically
chars = sorted(set(''.join(words)))

# dict of char-to-index
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi[end_char] = 0 # set char '.' at index 0

# dict of index-to-char
itos = {v:k for k,v in stoi.items()}

# set the vocab_size
vocab_size = len(itos)
print("Total vocab length",vocab_size)

Total vocab length 27


In [81]:
## Build dataset suitable for the model##
batch_size = 3

# function converts words into numerical of input sequences(contexts) and their next character targets
def build_dataset(words):
  X, Y = [], []
  for w in words: # iterarte over each words
    context = [0]* batch_size # initial value of context is [0, 0, 0] when batch_size is 3

    for c in w + end_char: # iterarte over each char in a word
      idx = stoi[c] # lookup the index for the char
      X.append(context) # initially it will be [0, 0, 0]
      Y.append(idx)
      context = context[1:] + [idx] # crop and append context

  X = torch.tensor(X) # 2D
  Y = torch.tensor(Y) # 1D
  return X, Y

In [82]:
## Build training,dev and test split from sample dataset ##

# shuffle the words
random.seed(42)
random.shuffle(words)

# get counts of train,dev,test
n1 = int(0.8*len(words)) # get count of 80% of words
n2 = int(0.9*len(words)) # get count of 90% of words

Xtr, Ytr = build_dataset(words[:n1]) # 80% training split
Xdev, Ydev = build_dataset(words[n1:n2]) # 10% dev split
Xte, Yte = build_dataset(words[:n2]) # 10% test split

print(Xtr.shape, Ytr.shape)

torch.Size([182625, 3]) torch.Size([182625])


In [83]:
## Create custom classe for neural net components ##

# Custom class to run layers sequentially
class Sequential:

  def __init__(self, layers):
    self.layers = layers

  def ___call__(self, x):
    for layer in self.layers:
      x = layer(x)
    self.out = x
    return self.out

  def parameters(self):
    return [p for layer in self.layers for p in layer.parameters()]

# Linear layer component
class Linear:

  def __init__(self, fan_in, fan_out, bias=True):
    self.weight = torch.randn((fan_in,fan_out)) / fan_in**0.5 # we could omit divison by 'fan_in**0.5', but keeping the Kaiming initialization (/ fan_in**0.5) is still a good practice and generally helps the training process.
    self.bais = torch.zeros(fan_out) if bias else None

  def __call__(self, x):
    self.out = x @ self.weight
    if self.bias is not None:
      self.out += self.bais
    return self.out

  def parameters(self):
    return [self.weight] + ([] if self.bias is None else [self.bais])

# BatchNormal layer component
class BatchNorm1d:

  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.momentum = momentum
    self.training = True
    self.gamma = torch.ones(dim) #similar weights in linear layer
    self.beta = torch.zeros(dim) #similar biases in linear layer
    # buffers (trained with a running 'momentum update')
    self.running_mean = torch.zeros(dim) # similar to previous bnmean_running
    self.running_var = torch.ones(dim) # similar to previous bnstd_running

  def __call__(self, x):
    # calculate the forward pass
    if self.training:
      if x.ndim == 2: # for 2D input
        dim = 0
      elif x.ndim == 3: # for 3D input
        dim = (0,1)
      xmean = x.mean(dim, keepdim=True) # batch mean
      xvar = x.var(dim, keepdim=True) # batch variance
    else:
      xmean = self.running_mean
      xvar = self.running_var
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    # update the buffers (similar how we updated bnmean_running and bnstd_running)
    if self.training:
      with torch.no_grad():
        self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * xmean
        self.running_var = (1 - self.momentum) * self.running_var + self.momentum * xvar
    return self.out

  def parameters(self):
    return [self.gamma, self.beta]

# Tanh component
class Tanh:

  def ___call__(self, x):
    self.out = torch.tanh(x)
    return self.out

  def parameters(self):
    return []

# Embedding component
class Embedding:

  def __init_(self, num_embeddings, embedding_dim):
    self.weight = torch.randn((num_embeddings, embedding_dim))

  def __call__(self, ix):
    self.out = self.weight[ix]
    return self.out

  def parameters(self):
    return [self.weight]

# Flattern component
class FlattenConsecutive:

  def __init__(self, ngram_dim):
    self.ngram_dim = ngram_dim

  def __call__(self, x):
    B, T, C = x.shape #eg: 32, 8, 10

    ## here we reshape x so that subsequent layer gets a grouped(ngram like)input
    x = x.view(B, T//self.ngram_dim, C*self.n)
    if x.shape[1] == 1: # This checks if the second dimension of the reshaped tensor is 1. This happens when the original sequence length T is equal to n (i.e T//n = 1)
      x = x.squeeze(1) # If the second dimension is 1, this line removes that dimension using squeeze(1).
    self.out = x
    return self.out

  def parameters(self):
    return []